# Advanced Problems with Solutions: Custom Sequences, Operators, and Mutable Polygon Sequences

This notebook contains advanced practice problems for:

- custom sequence protocols: `__len__`, `__getitem__`, `__setitem__`, `__delitem__`
- operator overloading: `+`, `+=`, `*`, `*=`, reflected operators
- correct in-place mutation semantics
- robust validation and conversion of sequence elements
- implementing a mutable sequence of geometric `Point` objects
- slice assignment, extended slices, deletion, append, extend, insert, pop, and membership

Each problem includes a complete solution and executable tests.


## Shared Testing Helpers

Run this cell first. It defines a tiny assertion helper so that the exercises can be checked without external dependencies.


In [1]:
from math import isclose, hypot
import numbers
from collections.abc import Iterable, MutableSequence

def assert_raises(exc_type, fn, *args, **kwargs):
    try:
        fn(*args, **kwargs)
    except exc_type:
        return True
    except Exception as ex:
        raise AssertionError(f"Expected {exc_type.__name__}, got {type(ex).__name__}: {ex}") from ex
    else:
        raise AssertionError(f"Expected {exc_type.__name__}, but no exception was raised")


## Problem 1 — A Production-Quality Addable and Repeatable Sequence

Implement `CheckedList`, a small mutable sequence wrapper that stores values in a private list and supports:

1. `len(obj)`, indexing, slicing, iteration, and `repr(obj)`
2. `obj + other`, where `other` may be another `CheckedList` or any iterable
3. `obj += other`, mutating the original object and preserving its identity
4. `obj * n`, `n * obj`, and `obj *= n`
5. Pythonic type behavior:
   - invalid repetition counts should naturally raise `TypeError`
   - unsupported concatenation should return `NotImplemented` where appropriate
6. optional element validation through a predicate passed to `__init__`

Best-practice requirement: avoid duplicating conversion logic between `__add__`, `__iadd__`, and `extend`.


### Solution 1

Key design choices:

- `__add__` returns a new object.
- `__iadd__` mutates `self` and returns `self`.
- `__mul__` returns a new object.
- `__imul__` mutates `self` and returns `self`.
- `__radd__` and `__rmul__` support reflected operations where they make sense.


In [2]:
class CheckedList:
    def __init__(self, values=(), *, validator=None):
        self._validator = validator
        self._items = []
        self.extend(values)

    def _coerce_many(self, values):
        try:
            values = list(values)
        except TypeError:
            raise TypeError("right operand must be iterable") from None

        if self._validator is not None:
            bad = [x for x in values if not self._validator(x)]
            if bad:
                raise TypeError(f"invalid element(s): {bad!r}")
        return values

    def __repr__(self):
        return f"{type(self).__name__}({self._items!r})"

    def __len__(self):
        return len(self._items)

    def __iter__(self):
        return iter(self._items)

    def __getitem__(self, index):
        result = self._items[index]
        if isinstance(index, slice):
            return type(self)(result, validator=self._validator)
        return result

    def __eq__(self, other):
        if isinstance(other, CheckedList):
            return self._items == other._items
        return self._items == other

    def extend(self, values):
        self._items.extend(self._coerce_many(values))

    def __add__(self, other):
        try:
            values = self._coerce_many(other._items if isinstance(other, CheckedList) else other)
        except TypeError:
            return NotImplemented
        return type(self)(self._items + values, validator=self._validator)

    def __radd__(self, other):
        try:
            left_values = self._coerce_many(other)
        except TypeError:
            return NotImplemented
        return type(self)(left_values + self._items, validator=self._validator)

    def __iadd__(self, other):
        values = other._items if isinstance(other, CheckedList) else other
        self.extend(values)
        return self

    def __mul__(self, n):
        return type(self)(self._items * n, validator=self._validator)

    def __rmul__(self, n):
        return self * n

    def __imul__(self, n):
        self._items *= n
        return self


In [3]:
# Tests for Solution 1
a = CheckedList([1, 2, 3], validator=lambda x: isinstance(x, int))
b = CheckedList([4, 5], validator=lambda x: isinstance(x, int))

assert len(a) == 3
assert a[1] == 2
assert list(a[1:]) == [2, 3]

c = a + b
assert c == [1, 2, 3, 4, 5]
assert a == [1, 2, 3]

old_id = id(a)
a += [6, 7]
assert id(a) == old_id
assert a == [1, 2, 3, 6, 7]

assert (CheckedList([1, 2]) * 3) == [1, 2, 1, 2, 1, 2]
assert (3 * CheckedList([1, 2])) == [1, 2, 1, 2, 1, 2]

x = CheckedList([9])
old_id = id(x)
x *= 4
assert id(x) == old_id
assert x == [9, 9, 9, 9]

assert_raises(TypeError, lambda: CheckedList([1], validator=lambda x: isinstance(x, int)) + object())
assert_raises(TypeError, CheckedList, [1, "bad"], validator=lambda x: isinstance(x, int))

print("Solution 1 passed.")


Solution 1 passed.


## Problem 2 — Detecting Correct In-Place Semantics

Create a function `audit_sequence_ops(seq_factory)` that receives a zero-argument factory returning a sequence-like object with values `[1, 2, 3]`.

The function must test and return a report dictionary with these boolean keys:

- `"add_returns_new_object"`
- `"iadd_preserves_identity"`
- `"mul_returns_new_object"`
- `"imul_preserves_identity"`
- `"slice_returns_same_type"`

Use it to audit `CheckedList`.


### Solution 2

The point of this exercise is not just to test results, but to test object identity. For mutable sequence types, `+=` and `*=` should generally mutate the original object and return it.


In [4]:
def audit_sequence_ops(seq_factory):
    seq = seq_factory()
    added = seq + [4]
    add_returns_new_object = id(added) != id(seq) and list(added) == [1, 2, 3, 4]

    seq = seq_factory()
    before = id(seq)
    seq += [4]
    iadd_preserves_identity = id(seq) == before and list(seq) == [1, 2, 3, 4]

    seq = seq_factory()
    multiplied = seq * 2
    mul_returns_new_object = id(multiplied) != id(seq) and list(multiplied) == [1, 2, 3, 1, 2, 3]

    seq = seq_factory()
    before = id(seq)
    seq *= 2
    imul_preserves_identity = id(seq) == before and list(seq) == [1, 2, 3, 1, 2, 3]

    seq = seq_factory()
    slice_returns_same_type = isinstance(seq[1:], type(seq)) and list(seq[1:]) == [2, 3]

    return {
        "add_returns_new_object": add_returns_new_object,
        "iadd_preserves_identity": iadd_preserves_identity,
        "mul_returns_new_object": mul_returns_new_object,
        "imul_preserves_identity": imul_preserves_identity,
        "slice_returns_same_type": slice_returns_same_type,
    }


In [5]:
report = audit_sequence_ops(lambda: CheckedList([1, 2, 3]))
assert all(report.values()), report
report


{'add_returns_new_object': True,
 'iadd_preserves_identity': True,
 'mul_returns_new_object': True,
 'imul_preserves_identity': True,
 'slice_returns_same_type': True}

## Problem 3 — A Robust `Point` Type

Implement a `Point` class suitable for use inside a custom sequence.

Requirements:

1. Accept exactly two real numeric coordinates.
2. Reject complex numbers, strings, and other non-real values.
3. Behave like a length-2 immutable sequence:
   - `len(p) == 2`
   - `p[0]`, `p[1]`
   - tuple unpacking works
4. Provide useful `repr`, equality, and hashing.
5. Provide a class method `from_value(value)` that accepts:
   - an existing `Point`
   - any two-item iterable of real numbers

Best-practice requirement: centralize conversion logic in `Point.from_value`.


### Solution 3

`Point` is immutable and hashable, while `Polygon` later will be mutable and unhashable.


In [6]:
class Point:
    __slots__ = ("_x", "_y")

    def __init__(self, x, y):
        if not isinstance(x, numbers.Real) or not isinstance(y, numbers.Real):
            raise TypeError("Point coordinates must be real numbers")
        self._x = x
        self._y = y

    @classmethod
    def from_value(cls, value):
        if isinstance(value, cls):
            return value

        try:
            x, y = value
        except TypeError:
            raise TypeError("Point value must be a Point or a two-item iterable") from None
        except ValueError:
            raise TypeError("Point value must contain exactly two items") from None

        return cls(x, y)

    @property
    def x(self):
        return self._x

    @property
    def y(self):
        return self._y

    def __len__(self):
        return 2

    def __iter__(self):
        yield self._x
        yield self._y

    def __getitem__(self, index):
        return (self._x, self._y)[index]

    def __repr__(self):
        return f"Point(x={self._x!r}, y={self._y!r})"

    def __eq__(self, other):
        try:
            other = Point.from_value(other)
        except TypeError:
            return False
        return (self.x, self.y) == (other.x, other.y)

    def __hash__(self):
        return hash((self._x, self._y))


In [7]:
# Tests for Solution 3
p = Point(1, 2.5)
assert len(p) == 2
assert p[0] == 1
assert p[1] == 2.5

x, y = p
assert (x, y) == (1, 2.5)

assert Point.from_value(p) is p
assert Point.from_value([3, 4]) == Point(3, 4)
assert hash(Point(1, 2)) == hash(Point(1, 2))

assert_raises(TypeError, Point, 1 + 2j, 3)
assert_raises(TypeError, Point.from_value, [1, 2, 3])
assert_raises(TypeError, Point.from_value, 10)

print("Solution 3 passed.")


Solution 3 passed.


## Problem 4 — Implement a Full Mutable `Polygon` Sequence

Implement `Polygon` as a mutable sequence of `Point` objects by subclassing `collections.abc.MutableSequence`.

Requirements:

1. Constructor accepts zero or more point-like values.
2. `repr(polygon)` should be clear and reconstructive.
3. Support:
   - `len(poly)`
   - indexing and slicing
   - iteration
   - membership with point-like values
   - `poly[i] = point_like`
   - simple slice assignment: `poly[i:j] = iterable_of_point_like`
   - extended slice assignment: `poly[i:j:k] = iterable_of_point_like`
   - deletion by index and slice
   - `insert`, `append`, `extend`, `pop`, `remove`
4. Slices should return a new `Polygon`, not a raw list.
5. Mutation methods should always store real `Point` instances.

Best-practice requirement: delegate common point conversion to a private helper.


### Solution 4

Subclassing `MutableSequence` reduces boilerplate: once `__len__`, `__getitem__`, `__setitem__`, `__delitem__`, and `insert` are correct, the ABC supplies methods such as `append`, `extend`, `pop`, and `remove`.


In [8]:
class Polygon(MutableSequence):
    def __init__(self, *points):
        self._points = []
        self.extend(points)

    @staticmethod
    def _coerce_point(value):
        return Point.from_value(value)

    @classmethod
    def _coerce_points(cls, values):
        try:
            return [cls._coerce_point(value) for value in values]
        except TypeError:
            raise TypeError("expected an iterable of point-like values") from None

    def __repr__(self):
        args = ", ".join(repr(point) for point in self._points)
        return f"Polygon({args})"

    def __len__(self):
        return len(self._points)

    def __iter__(self):
        return iter(self._points)

    def __getitem__(self, index):
        result = self._points[index]
        if isinstance(index, slice):
            return type(self)(*result)
        return result

    def __setitem__(self, index, value):
        if isinstance(index, slice):
            self._points[index] = self._coerce_points(value)
        else:
            self._points[index] = self._coerce_point(value)

    def __delitem__(self, index):
        del self._points[index]

    def insert(self, index, value):
        self._points.insert(index, self._coerce_point(value))

    def __contains__(self, value):
        try:
            point = self._coerce_point(value)
        except TypeError:
            return False
        return point in self._points

    def __eq__(self, other):
        if isinstance(other, Polygon):
            return self._points == other._points
        try:
            return self._points == self._coerce_points(other)
        except TypeError:
            return False


In [9]:
# Tests for Solution 4
poly = Polygon((0, 0), Point(1, 1), [2, 2])
assert len(poly) == 3
assert poly[0] == Point(0, 0)
assert isinstance(poly[1:], Polygon)
assert list(poly[1:]) == [Point(1, 1), Point(2, 2)]

assert (1, 1) in poly
assert Point(2, 2) in poly
assert "not a point" not in poly

poly[0] = [10, 10]
assert poly[0] == Point(10, 10)

poly[1:3] = [(20, 20), Point(30, 30), [40, 40]]
assert list(poly) == [Point(10, 10), Point(20, 20), Point(30, 30), Point(40, 40)]

poly[::2] = [(100, 100), (300, 300)]
assert list(poly) == [Point(100, 100), Point(20, 20), Point(300, 300), Point(40, 40)]

assert_raises(ValueError, lambda: poly.__setitem__(slice(None, None, 2), [(1, 1)]))

del poly[1]
assert list(poly) == [Point(100, 100), Point(300, 300), Point(40, 40)]

poly.insert(1, (200, 200))
poly.append((500, 500))
assert poly.pop() == Point(500, 500)
assert list(poly) == [Point(100, 100), Point(200, 200), Point(300, 300), Point(40, 40)]

print("Solution 4 passed.")


Solution 4 passed.


## Problem 5 — Concatenation and In-Place Extension for `Polygon`

Extend `Polygon` so it supports:

1. `p1 + p2`, returning a new `Polygon`
2. `p1 + iterable_of_point_like`, returning a new `Polygon`
3. `iterable_of_point_like + p1`, returning a new `Polygon`
4. `p1 += p2`, mutating `p1`
5. `p1 += iterable_of_point_like`, mutating `p1`

Best-practice requirements:

- `+` must not mutate either operand.
- `+=` must preserve identity.
- Use `NotImplemented` for unsupported binary operations where possible.


### Solution 5

We redefine `Polygon` with the same sequence behavior plus `__add__`, `__radd__`, and `__iadd__`.


In [10]:
class Polygon(MutableSequence):
    def __init__(self, *points):
        self._points = []
        self.extend(points)

    @staticmethod
    def _coerce_point(value):
        return Point.from_value(value)

    @classmethod
    def _coerce_points(cls, values):
        if isinstance(values, Polygon):
            return list(values._points)
        try:
            return [cls._coerce_point(value) for value in values]
        except TypeError:
            raise TypeError("expected an iterable of point-like values") from None

    def __repr__(self):
        args = ", ".join(repr(point) for point in self._points)
        return f"Polygon({args})"

    def __len__(self):
        return len(self._points)

    def __iter__(self):
        return iter(self._points)

    def __getitem__(self, index):
        result = self._points[index]
        if isinstance(index, slice):
            return type(self)(*result)
        return result

    def __setitem__(self, index, value):
        if isinstance(index, slice):
            self._points[index] = self._coerce_points(value)
        else:
            self._points[index] = self._coerce_point(value)

    def __delitem__(self, index):
        del self._points[index]

    def insert(self, index, value):
        self._points.insert(index, self._coerce_point(value))

    def __contains__(self, value):
        try:
            point = self._coerce_point(value)
        except TypeError:
            return False
        return point in self._points

    def __eq__(self, other):
        if isinstance(other, Polygon):
            return self._points == other._points
        try:
            return self._points == self._coerce_points(other)
        except TypeError:
            return False

    def __add__(self, other):
        try:
            points = self._coerce_points(other)
        except TypeError:
            return NotImplemented
        return type(self)(*self._points, *points)

    def __radd__(self, other):
        try:
            points = self._coerce_points(other)
        except TypeError:
            return NotImplemented
        return type(self)(*points, *self._points)

    def __iadd__(self, other):
        self.extend(self._coerce_points(other))
        return self


In [11]:
# Tests for Solution 5
p1 = Polygon((0, 0), (1, 1))
p2 = Polygon((2, 2), (3, 3))

p3 = p1 + p2
assert isinstance(p3, Polygon)
assert list(p3) == [Point(0, 0), Point(1, 1), Point(2, 2), Point(3, 3)]
assert list(p1) == [Point(0, 0), Point(1, 1)]
assert list(p2) == [Point(2, 2), Point(3, 3)]

p4 = p1 + [(4, 4), (5, 5)]
assert list(p4) == [Point(0, 0), Point(1, 1), Point(4, 4), Point(5, 5)]

p5 = [(9, 9)] + p1
assert list(p5) == [Point(9, 9), Point(0, 0), Point(1, 1)]

old_id = id(p1)
p1 += [(6, 6), Point(7, 7)]
assert id(p1) == old_id
assert list(p1) == [Point(0, 0), Point(1, 1), Point(6, 6), Point(7, 7)]

assert_raises(TypeError, lambda: p1 + object())

print("Solution 5 passed.")


Solution 5 passed.


## Problem 6 — Geometry Methods on a Mutable Sequence

Add these methods/properties to `Polygon`:

1. `.closed_points`: returns the polygon points with the first point repeated at the end, but only if the polygon has at least one point.
2. `.perimeter`: total distance around the closed polygon.
3. `.area`: signed shoelace area as a positive number.
4. `.is_closed`: always `True` for geometric calculations with at least 3 points, even if the first point is not physically repeated.
5. Validation:
   - perimeter and area require at least 3 points
   - repeated explicit closing point should not double-count the first edge

Best-practice requirement: keep storage simple. Do not physically append a closing point just to compute perimeter or area.


### Solution 6

A common bug is to mutate the polygon by appending the first point to the end. That makes later calculations wrong. Instead, compute a temporary closed view.


In [12]:
class GeometricPolygon(Polygon):
    @property
    def _unique_boundary_points(self):
        points = list(self)
        if len(points) >= 2 and points[0] == points[-1]:
            points = points[:-1]
        return points

    def _require_polygon(self):
        if len(self._unique_boundary_points) < 3:
            raise ValueError("a polygon needs at least three distinct boundary points")

    @property
    def closed_points(self):
        points = self._unique_boundary_points
        if not points:
            return []
        return points + [points[0]]

    @property
    def is_closed(self):
        return len(self._unique_boundary_points) >= 3

    @property
    def perimeter(self):
        self._require_polygon()
        total = 0
        pts = self.closed_points
        for p1, p2 in zip(pts, pts[1:]):
            total += hypot(p2.x - p1.x, p2.y - p1.y)
        return total

    @property
    def area(self):
        self._require_polygon()
        pts = self.closed_points
        total = 0
        for p1, p2 in zip(pts, pts[1:]):
            total += p1.x * p2.y - p2.x * p1.y
        return abs(total) / 2


In [13]:
# Tests for Solution 6
triangle = GeometricPolygon((0, 0), (3, 0), (0, 4))
assert triangle.closed_points == [Point(0, 0), Point(3, 0), Point(0, 4), Point(0, 0)]
assert triangle.is_closed is True
assert isclose(triangle.perimeter, 12.0)
assert isclose(triangle.area, 6.0)

explicitly_closed = GeometricPolygon((0, 0), (3, 0), (0, 4), (0, 0))
assert explicitly_closed.closed_points == triangle.closed_points
assert isclose(explicitly_closed.perimeter, 12.0)
assert isclose(explicitly_closed.area, 6.0)

square = GeometricPolygon((0, 0), (2, 0), (2, 2), (0, 2))
assert isclose(square.perimeter, 8.0)
assert isclose(square.area, 4.0)

line = GeometricPolygon((0, 0), (1, 1))
assert_raises(ValueError, lambda: line.perimeter)
assert_raises(ValueError, lambda: line.area)

print("Solution 6 passed.")


Solution 6 passed.


## Challenge Problem — Transaction-Safe Slice Assignment

Implement `safe_replace(poly, index_or_slice, values)`.

The function should replace part of a polygon, but with transaction safety:

- If conversion of the replacement values fails, the original polygon must remain unchanged.
- It must support both index assignment and slice assignment.
- It must return the polygon itself after successful mutation.

This is especially important for mutable containers: partial mutation followed by an exception can leave an object in an invalid or surprising state.


### Challenge Solution

For index assignment, convert the single replacement before mutating.

For slice assignment, convert the entire replacement list before mutating. This prevents half-applied changes.


In [14]:
def safe_replace(poly, index_or_slice, values):
    if isinstance(index_or_slice, slice):
        converted = poly._coerce_points(values)
        poly[index_or_slice] = converted
    else:
        converted = poly._coerce_point(values)
        poly[index_or_slice] = converted
    return poly


In [15]:
# Tests for Challenge Solution
poly = Polygon((0, 0), (1, 1), (2, 2))
safe_replace(poly, 1, (10, 10))
assert list(poly) == [Point(0, 0), Point(10, 10), Point(2, 2)]

safe_replace(poly, slice(1, 3), [(20, 20), (30, 30)])
assert list(poly) == [Point(0, 0), Point(20, 20), Point(30, 30)]

before = list(poly)
assert_raises(TypeError, safe_replace, poly, slice(1, 3), [(99, 99), ("bad", 100)])
assert list(poly) == before

print("Challenge solution passed.")


Challenge solution passed.


## Summary Checklist

A high-quality custom mutable sequence should usually:

- centralize validation and conversion
- return `NotImplemented` from binary operators when appropriate
- ensure `+` and `*` create new objects
- ensure `+=` and `*=` preserve identity when implemented
- return the custom sequence type for slices
- avoid partial mutation on failed validation
- delegate standard mutable sequence behavior to `MutableSequence` when possible
- keep internal storage simple and enforce invariants at boundaries
